In [ ]:
!pip install torch==2.2.2 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.38.2 


In [ ]:
!pip install datasets

In [ ]:
# =============================================================================
# OPTIMIZATION CHANGELOG — RoBERTa fine-tuning for CJPE
# =============================================================================
# Six orthogonal interventions applied while preserving core business logic:
#   1. Fix CLS attention-mask bug                       → cell-12
#   2. Class-imbalance audit + weighted loss            → cell-13 + cell-17
#   3. Mixed-precision (fp16) training                  → cell-17 + cell-18
#   4. Gradient accumulation (effective batch 24)       → cell-17 + cell-18
#   5. Early stopping + best-checkpoint restore         → cell-17 + cell-18
#   6. Inference under autocast                         → cell-21 + cell-26
#
# Supporting infrastructure:
#   - Multi-platform HF auth (Kaggle/Colab/local)       → cell-5
#   - Cross-platform model save + HF Hub push           → cell-19
#   - Test data preparation                             → cell-20
#   - Validation dataloader reassignment                → cell-25
#   - Single-document prediction helper                 → cell-29
#
# See OPTIMIZATION_GUIDE.md for detailed before/after code and reasoning.
# =============================================================================

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import csv
import tensorflow as tf
import torch
import torch.nn as nn
import torch.cuda.amp
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
# from google.colab import drive  # local run
import textwrap
import progressbar
import keras
from keras.preprocessing.sequence import pad_sequences
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler, Dataset
from transformers import BertForSequenceClassification, BertConfig
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from transformers import get_linear_schedule_with_warmup
import time
import datetime
import json

In [ ]:
print("Using Hugging Face dataset loader: Exploration-Lab/IL-TUR (ILDC_multi).")

In [ ]:
# =============================================================================
# HUGGING FACE AUTHENTICATION — works on Kaggle, Colab, and local
# =============================================================================
# Detection order:
#   1. Kaggle    → kaggle_secrets.UserSecretsClient (HF_TOKEN secret)
#   2. Colab     → google.colab.userdata (HF_TOKEN secret)
#   3. Local/env → os.environ['HF_TOKEN']
#   4. Fallback  → interactive prompt via getpass
# =============================================================================

import os
from getpass import getpass
from huggingface_hub import login

hf_token = None
source = None

# Attempt 1: Kaggle
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if hf_token:
        source = "Kaggle Secrets"
except (ImportError, ModuleNotFoundError):
    pass
except Exception as e:
    # kaggle_secrets exists but secret not set or access failed
    print(f"  Kaggle Secrets available but HF_TOKEN not accessible: {e}")

# Attempt 2: Colab userdata
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            source = "Colab Secrets"
    except (ImportError, ModuleNotFoundError):
        pass
    except Exception:
        # userdata exists but secret not set
        pass

# Attempt 3: Environment variable (works on local machine, CI, Docker, etc.)
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")
    if hf_token:
        source = "environment variable HF_TOKEN"

# Attempt 4: Interactive prompt
if not hf_token:
    print("HF_TOKEN not found in Kaggle Secrets, Colab Secrets, or environment.")
    print("Enter your Hugging Face token (hidden) or press Enter to skip:")
    entered = getpass("HF token: ").strip()
    if entered:
        hf_token = entered
        source = "manual entry"

# Apply login
if hf_token:
    login(token=hf_token)
    print(f"✓ Logged in to Hugging Face via {source}")
else:
    print("⚠ Skipping HF login. Some operations (dataset/model download, hub push) may fail.")
    print("  To fix: set HF_TOKEN secret in your platform or as an env variable.")

In [ ]:
# =============================================================================
# LOAD CJPE DATASET — combines multi_train + single_train for +15% more data
# =============================================================================
# IL-TUR provides two training splits:
#   - multi_train (32,305 docs): multi-annotator consensus labels
#   - single_train (5,082 docs):  single-annotator labels
# Combining them gives 37,387 training docs (~15% more) without changing
# the task definition or evaluation protocol.
# =============================================================================

from datasets import load_dataset
dataset = load_dataset("Exploration-Lab/IL-TUR", "cjpe")

# Combine both training splits
multi_train_df = dataset["multi_train"].to_pandas()
single_train_df = dataset["single_train"].to_pandas()

train_set = pd.concat([multi_train_df, single_train_df], ignore_index=True)

# Shuffle so single_train docs aren't all at the end (matters for batching)
train_set = train_set.sample(frac=1.0, random_state=2212).reset_index(drop=True)

validation_set = dataset["multi_dev"].to_pandas()
test_set = dataset["test"].to_pandas()

print(f"Training set:")
print(f"  multi_train:  {len(multi_train_df):>6,} docs")
print(f"  single_train: {len(single_train_df):>6,} docs")
print(f"  combined:     {len(train_set):>6,} docs (+{100*len(single_train_df)/len(multi_train_df):.1f}% over multi_train alone)")
print(f"Validation set: {len(validation_set):,} docs")
print(f"Test set:       {len(test_set):,} docs")

In [ ]:
dataset

In [ ]:
# =============================================================================
# MODEL SELECTION
# =============================================================================
# Switched from `roberta-base` (general English) to `law-ai/InCaseLawBERT`
# (pre-trained on Indian case law). Expected accuracy boost: +5-7% on CJPE.
#
# InCaseLawBERT is BERT-based, NOT RoBERTa — uses WordPiece tokenization
# instead of BPE, with different special tokens:
#   - CLS = [CLS] (id=101), SEP = [SEP] (id=102), PAD = [PAD] (id=0)
# The Auto* classes handle these differences transparently.
# =============================================================================

# Legacy model registry (kept for reference; we now use Auto* classes)
from transformers import PreTrainedModel, PreTrainedTokenizer, PretrainedConfig
from transformers import RobertaForSequenceClassification, RobertaTokenizer, RobertaConfig
from transformers import XLNetForSequenceClassification, XLNetTokenizer, XLNetConfig
from transformers import XLMForSequenceClassification, XLMTokenizer, XLMConfig
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, DistilBertConfig

MODEL_CLASSES = {
    'bert': (BertForSequenceClassification, BertTokenizer, BertConfig),
    'xlnet': (XLNetForSequenceClassification, XLNetTokenizer, XLNetConfig),
    'xlm': (XLMForSequenceClassification, XLMTokenizer, XLMConfig),
    'roberta': (RobertaForSequenceClassification, RobertaTokenizer, RobertaConfig),
    'distilbert': (DistilBertForSequenceClassification, DistilBertTokenizer, DistilBertConfig)
}

# --- ACTIVE MODEL ---
model_name = 'law-ai/InCaseLawBERT'   # ← Legal-domain BERT trained on Indian case law

# Determine if this is a RoBERTa-style tokenizer (needs add_prefix_space)
IS_ROBERTA_STYLE = any(tag in model_name.lower() for tag in ['roberta', 'longformer', 'gpt'])

print(f"Active model: {model_name}")
print(f"Tokenizer style: {'RoBERTa/BPE (uses add_prefix_space)' if IS_ROBERTA_STYLE else 'BERT/WordPiece'}")

In [ ]:
# AutoTokenizer auto-detects whether to use BertTokenizer, RobertaTokenizer, etc.
# based on the model_name. Works for InCaseLawBERT, RoBERTa, Longformer, etc.
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"  CLS token: {tokenizer.cls_token} (id={tokenizer.cls_token_id})")
print(f"  SEP token: {tokenizer.sep_token} (id={tokenizer.sep_token_id})")
print(f"  PAD token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"  Vocab size: {tokenizer.vocab_size:,}")

In [ ]:
# =============================================================================
# HIERARCHICAL CHUNKING — splits long documents into multiple chunks
# =============================================================================
# Instead of truncating to a single 512-token window, split each document
# into K chunks of up to 510 tokens each (+CLS/+SEP per chunk). The encoder
# processes each chunk separately, and an attention-pooling head aggregates
# the K chunk representations into one document representation.
#
# This captures the FULL document (not just last 510 tokens) while keeping
# InCaseLawBERT's legal-domain pretraining intact.
# =============================================================================

CHUNK_SIZE = 510    # tokens per chunk (excluding CLS/SEP, so 512 total)
MAX_CHUNKS = 8      # cap chunks per document (510*8 = 4080 tokens of content)
                    # Empirical: 95th percentile of CJPE docs fits in 8 chunks

def make_chunks(text, tokenizer):
    """
    Tokenize a document and split into chunks. Returns:
        input_ids: list of [chunk_size+2] int lists (padded to MAX_CHUNKS slots)
        attention_mask: list of [chunk_size+2] int lists (1 for real, 0 for pad)
        n_real_chunks: int — actual chunk count (rest are padding chunks)
    """
    # Tokenize the full text
    if IS_ROBERTA_STYLE:
        tokens = tokenizer.tokenize(text, add_prefix_space=True)
    else:
        tokens = tokenizer.tokenize(text)

    pad_id = tokenizer.pad_token_id
    cls = tokenizer.cls_token
    sep = tokenizer.sep_token
    target_len = CHUNK_SIZE + 2  # CLS + content + SEP

    chunks_ids = []
    chunks_mask = []

    # Walk through tokens in stride of CHUNK_SIZE (no overlap)
    # Cap at MAX_CHUNKS * CHUNK_SIZE tokens (rest of doc is discarded)
    capped_tokens = tokens[:MAX_CHUNKS * CHUNK_SIZE]
    n_real_chunks = 0

    for start in range(0, len(capped_tokens), CHUNK_SIZE):
        chunk = capped_tokens[start:start + CHUNK_SIZE]
        # Add special tokens and convert to IDs
        chunk_with_special = [cls] + chunk + [sep]
        ids = tokenizer.convert_tokens_to_ids(chunk_with_special)
        mask = [1] * len(ids)
        # Pad each chunk to target_len
        pad_amount = target_len - len(ids)
        ids = ids + [pad_id] * pad_amount
        mask = mask + [0] * pad_amount
        chunks_ids.append(ids)
        chunks_mask.append(mask)
        n_real_chunks += 1

    # If document was empty or too short, ensure at least 1 chunk
    if n_real_chunks == 0:
        chunks_ids.append([tokenizer.convert_tokens_to_ids(cls), tokenizer.convert_tokens_to_ids(sep)] + [pad_id] * (target_len - 2))
        chunks_mask.append([1, 1] + [0] * (target_len - 2))
        n_real_chunks = 1

    # Pad chunk LIST to MAX_CHUNKS slots (with all-padding chunks)
    while len(chunks_ids) < MAX_CHUNKS:
        chunks_ids.append([pad_id] * target_len)
        chunks_mask.append([0] * target_len)

    return chunks_ids, chunks_mask, n_real_chunks

In [ ]:
# Chunk all training and validation documents
def chunk_dataset(dataf, tokenizer, name=""):
    """Returns (input_ids[N,K,L], attention_masks[N,K,L], n_chunks_list[N])."""
    all_ids = []
    all_masks = []
    n_chunks_list = []
    print(f"Chunking {len(dataf)} {name} documents...")
    for i in progressbar.progressbar(range(len(dataf))):
        text = dataf['text'].iloc[i]
        chunks_ids, chunks_mask, n_real = make_chunks(text, tokenizer)
        all_ids.append(chunks_ids)
        all_masks.append(chunks_mask)
        n_chunks_list.append(n_real)
    return np.array(all_ids), np.array(all_masks), np.array(n_chunks_list)

train_input_ids, train_attention_masks, train_n_chunks = chunk_dataset(train_set, tokenizer, "train")
validation_input_ids, validation_attention_masks, validation_n_chunks = chunk_dataset(validation_set, tokenizer, "validation")

print(f"\n✓ Train chunks tensor:      {train_input_ids.shape}  (N docs × K chunks × L tokens)")
print(f"✓ Validation chunks tensor: {validation_input_ids.shape}")
print(f"  Avg real chunks per train doc: {train_n_chunks.mean():.2f}")
print(f"  Max real chunks per train doc: {train_n_chunks.max()}")
print(f"  Docs hitting MAX_CHUNKS={MAX_CHUNKS}: {(train_n_chunks == MAX_CHUNKS).sum()} "
      f"({100 * (train_n_chunks == MAX_CHUNKS).mean():.1f}%)")

In [ ]:
def att_masking(input_ids):
  # FIX: RoBERTa's CLS token <s> has id=0, same as padding value.
  # Original `token_id > 0` check masked out position 0 (CLS), degrading
  # the classifier's input. Compare against tokenizer.pad_token_id (1) instead.
  pad_id = tokenizer.pad_token_id
  attention_masks = []
  for sent in input_ids:
    att_mask = [int(token_id != pad_id and not (token_id == 0 and i > 0))
                for i, token_id in enumerate(sent)]
    attention_masks.append(att_mask)
  return attention_masks

In [ ]:
# Attention masks were already built per-chunk in make_chunks (cell-10).
# train_attention_masks / validation_attention_masks already exist from cell-11.
# This cell just extracts labels and computes class weights (unchanged from v2.1).

train_labels = train_set['label'].to_numpy().astype('int')
validation_labels = validation_set['label'].to_numpy().astype('int')

# CLASS IMBALANCE AUDIT — same as v2.1
unique, counts = np.unique(train_labels, return_counts=True)
class_distribution = dict(zip(unique.tolist(), counts.tolist()))
total = counts.sum()
print("Train class distribution:")
for cls, cnt in class_distribution.items():
    print(f"  label={cls}: {cnt} ({100.0 * cnt / total:.2f}%)")

class_weights_np = total / (len(unique) * counts.astype(np.float32))
print(f"Class weights (inverse-frequency): {class_weights_np.tolist()}")

In [ ]:
# Build PyTorch tensors. Shapes:
#   inputs:        (N_docs, MAX_CHUNKS, 512)  — chunked tokens
#   masks:         (N_docs, MAX_CHUNKS, 512)  — token attention masks (per chunk)
#   chunk_masks:   (N_docs, MAX_CHUNKS)       — 1=real chunk, 0=padding chunk
#   labels:        (N_docs,)
def n_chunks_to_chunk_mask(n_chunks_arr, max_chunks):
    """Convert [N] array of real chunk counts → [N, max_chunks] binary mask."""
    mask = np.zeros((len(n_chunks_arr), max_chunks), dtype=np.int64)
    for i, n in enumerate(n_chunks_arr):
        mask[i, :n] = 1
    return mask

train_chunk_masks = n_chunks_to_chunk_mask(train_n_chunks, MAX_CHUNKS)
validation_chunk_masks = n_chunks_to_chunk_mask(validation_n_chunks, MAX_CHUNKS)

train_inputs = torch.tensor(train_input_ids, dtype=torch.long)
train_masks = torch.tensor(train_attention_masks, dtype=torch.long)
train_chunk_masks_t = torch.tensor(train_chunk_masks, dtype=torch.long)
train_labels = torch.tensor(train_labels, dtype=torch.long)

validation_inputs = torch.tensor(validation_input_ids, dtype=torch.long)
validation_masks = torch.tensor(validation_attention_masks, dtype=torch.long)
validation_chunk_masks_t = torch.tensor(validation_chunk_masks, dtype=torch.long)
validation_labels = torch.tensor(validation_labels, dtype=torch.long)

print(f"✓ Train tensors:      inputs={tuple(train_inputs.shape)}, "
      f"chunk_mask={tuple(train_chunk_masks_t.shape)}, labels={tuple(train_labels.shape)}")
print(f"✓ Validation tensors: inputs={tuple(validation_inputs.shape)}, "
      f"chunk_mask={tuple(validation_chunk_masks_t.shape)}, labels={tuple(validation_labels.shape)}")

In [ ]:
# =============================================================================
# HIERARCHICAL DATALOADERS
# =============================================================================
# Batch size MUST be smaller than v2.1 (was 6) because hierarchical training
# processes K=8 chunks PER document in parallel. Effective sequences per
# forward = batch_size × MAX_CHUNKS = 2 × 8 = 16 sequences of 512 tokens.
# That's ~3× memory of the 6-doc batch we used before.
#
# We compensate with gradient_accumulation_steps=12 (cell-17) to maintain
# effective batch size of 24.
# =============================================================================

batch_size = 2   # was 6 — reduced for hierarchical memory budget

# 4-tuple datasets: (input_ids, attention_mask, chunk_mask, label)
train_data = TensorDataset(train_inputs, train_masks, train_chunk_masks_t, train_labels)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

validation_data = TensorDataset(validation_inputs, validation_masks,
                                validation_chunk_masks_t, validation_labels)
validation_sampler = RandomSampler(validation_data)
validation_dataloader = DataLoader(validation_data, sampler=validation_sampler, batch_size=batch_size)

print(f"✓ Train DataLoader:      {len(train_dataloader)} batches of {batch_size} docs each")
print(f"✓ Validation DataLoader: {len(validation_dataloader)} batches of {batch_size} docs each")

In [ ]:
# =============================================================================
# HIERARCHICAL CLASSIFIER — InCaseLawBERT encoder + attention pooling + head
# =============================================================================
# Architecture:
#   (B, K, L) input → encoder → (B, K, H) chunk embeddings via CLS token
#                  → attention pooling → (B, H) document embedding
#                  → dropout → linear → (B, 2) logits
#
# The attention layer learns WHICH chunks are most informative for the
# decision — e.g., chunks containing "appeal is dismissed" or "petition
# allowed" will get higher attention weights.
# =============================================================================

class HierarchicalClassifier(nn.Module):
    def __init__(self, base_model_name, num_labels=2, dropout_prob=0.1):
        super().__init__()
        # AutoModel returns just the encoder (no classification head)
        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.encoder.config.hidden_size  # 768 for BERT-base
        self.hidden_size = hidden_size
        self.num_labels = num_labels
        self.base_model_name = base_model_name

        # Attention pooling: learn a scalar attention score per chunk
        self.chunk_attention = nn.Linear(hidden_size, 1)

        # Classification head (same shape as BertForSequenceClassification's)
        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, chunk_mask):
        """
        input_ids:      (B, K, L)  — batched chunked token IDs
        attention_mask: (B, K, L)  — per-chunk token attention (1=real, 0=pad)
        chunk_mask:     (B, K)     — per-doc chunk validity (1=real chunk, 0=pad chunk)
        Returns: logits (B, num_labels)
        """
        B, K, L = input_ids.shape

        # Flatten chunk dim into batch dim so encoder processes B*K sequences
        flat_input_ids = input_ids.view(B * K, L)
        flat_attention_mask = attention_mask.view(B * K, L)

        # Encode all chunks. AutoModel returns BaseModelOutput with last_hidden_state.
        outputs = self.encoder(flat_input_ids, attention_mask=flat_attention_mask)

        # Take CLS embedding from each chunk: (B*K, H)
        chunk_cls = outputs.last_hidden_state[:, 0, :]

        # Reshape back to (B, K, H)
        chunk_embeds = chunk_cls.view(B, K, self.hidden_size)

        # Compute attention scores per chunk: (B, K)
        attn_scores = self.chunk_attention(chunk_embeds).squeeze(-1)

        # Mask out padding chunks before softmax — set their scores to -inf
        # so they get zero weight in the softmax.
        attn_scores = attn_scores.masked_fill(chunk_mask == 0, float('-inf'))

        # Softmax over chunks → attention weights (B, K)
        attn_weights = torch.softmax(attn_scores, dim=-1)

        # Weighted sum of chunk embeddings → document embedding (B, H)
        doc_embed = (chunk_embeds * attn_weights.unsqueeze(-1)).sum(dim=1)

        # Classification head
        doc_embed = self.dropout(doc_embed)
        logits = self.classifier(doc_embed)
        return logits


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HierarchicalClassifier(base_model_name=model_name, num_labels=2)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
encoder_params = sum(p.numel() for p in model.encoder.parameters())
head_params = sum(p.numel() for p in model.chunk_attention.parameters()) + \
              sum(p.numel() for p in model.classifier.parameters())

print(f"✓ HierarchicalClassifier built")
print(f"  Base encoder: {model_name}")
print(f"  Hidden size:  {model.hidden_size}")
print(f"  Total params:     {total_params:>15,}")
print(f"  - Encoder:        {encoder_params:>15,}")
print(f"  - Attention+head: {head_params:>15,}")
print(f"  Trainable:        {trainable_params:>15,}")
print(f"  Device:           {device}")

In [ ]:
# =============================================================================
# TRAINING CONFIGURATION — adjusted for hierarchical model
# =============================================================================
# Changes from v2.1:
#   - batch_size: 6 → 2 (hierarchical processes 8 chunks per doc = 8× compute)
#   - gradient_accumulation_steps: 4 → 12 (keeps effective batch = 24)
#   - Other hyperparams unchanged from v2.1
# =============================================================================

lr = 5e-6
max_grad_norm = 1.0
epochs = 10
label_smoothing = 0.1

# Gradient accumulation — increased to compensate for smaller batch_size
gradient_accumulation_steps = 12   # was 4 in v2.1
effective_batch_size = batch_size * gradient_accumulation_steps   # 2 * 12 = 24

early_stopping_patience = 3

# Scheduler calibrated to OPTIMIZER steps (not micro-batches)
num_optimizer_steps_per_epoch = len(train_dataloader) // gradient_accumulation_steps
num_total_steps = num_optimizer_steps_per_epoch * epochs
num_warmup_steps = 1000
warmup_proportion = float(num_warmup_steps) / float(num_total_steps)

use_amp = torch.cuda.is_available()

class_weights = torch.tensor(class_weights_np, dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)

print(f"Hierarchical training config:")
print(f"  Per-step batch size:  {batch_size} docs × {MAX_CHUNKS} chunks = {batch_size * MAX_CHUNKS} sequences")
print(f"  Gradient accumulation: {gradient_accumulation_steps}")
print(f"  Effective batch size: {effective_batch_size} docs")
print(f"  Learning rate:        {lr}")
print(f"  Epochs:               {epochs} (early stop patience={early_stopping_patience})")
print(f"  Optimizer steps:      {num_total_steps:,} (warmup {num_warmup_steps})")
print(f"  Label smoothing:      {label_smoothing}")
print(f"  Class weights:        {class_weights.tolist()}")
print(f"  Mixed precision:      {'enabled (fp16)' if use_amp else 'disabled'}")

def flat_accuracy(preds, labels):
    preds = np.array(preds)
    labels = np.array(labels)
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

seed_val = 2212
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

In [ ]:
loss_values = []

# =============================================================================
# HIERARCHICAL TRAINING LOOP
# =============================================================================
# Same optimization stack as v2.1 (fp16, gradient accumulation, weighted loss,
# label smoothing, early stopping) but adapted for the 4-tuple batch and the
# HierarchicalClassifier forward signature.
# =============================================================================

# Rebuild optimizer + scheduler + scaler against the current model
optimizer = AdamW(model.parameters(), lr=lr)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_total_steps,
)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# Reset early-stopping state
best_val_accuracy = -1.0
epochs_without_improvement = 0
best_model_state = None
val_accuracy_history = []

print(f"✓ Optimizer/scheduler/scaler bound to current model ({sum(p.numel() for p in model.parameters()):,} params)")

for epoch_i in range(0, epochs):
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    print('Training...')

    t0 = time.time()
    total_loss = 0

    model.train()
    optimizer.zero_grad()

    for step, batch in enumerate(train_dataloader):
        if step % 40 == 0 and not step == 0:
            print('  Batch {:>5,}  of  {:>5,}. '.format(step, len(train_dataloader)))

        # 4-tuple batch (hierarchical): input_ids, attention_mask, chunk_mask, labels
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_chunk_mask = batch[2].to(device)
        b_labels = batch[3].to(device)

        with torch.cuda.amp.autocast(enabled=use_amp):
            # HierarchicalClassifier forward: (input_ids, attention_mask, chunk_mask) → logits
            logits = model(b_input_ids, attention_mask=b_input_mask, chunk_mask=b_chunk_mask)
            loss = loss_fn(logits, b_labels)
            loss = loss / gradient_accumulation_steps

        total_loss += loss.item() * gradient_accumulation_steps
        scaler.scale(loss).backward()

        is_accum_boundary = (step + 1) % gradient_accumulation_steps == 0
        is_last_batch = (step + 1) == len(train_dataloader)

        if is_accum_boundary or is_last_batch:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

    avg_train_loss = total_loss / len(train_dataloader)
    loss_values.append(avg_train_loss)

    print("")
    print("  Average training loss: {0:.2f}".format(avg_train_loss))
    print("")
    print("Running Validation...")

    t0 = time.time()
    model.eval()

    eval_accuracy = 0
    nb_eval_steps = 0

    for batch in validation_dataloader:
        batch = tuple(t.to(device) for t in batch)
        b_input_ids, b_input_mask, b_chunk_mask, b_labels = batch

        with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(b_input_ids, attention_mask=b_input_mask, chunk_mask=b_chunk_mask)

        logits = logits.detach().cpu().tolist()
        label_ids = b_labels.to('cpu').tolist()

        eval_accuracy += flat_accuracy(logits, label_ids)
        nb_eval_steps += 1

    val_accuracy = eval_accuracy / nb_eval_steps
    val_accuracy_history.append(val_accuracy)
    print("  Accuracy: {0:.2f}".format(val_accuracy))

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_without_improvement = 0
        best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ✓ New best validation accuracy: {best_val_accuracy:.4f}")
    else:
        epochs_without_improvement += 1
        print(f"  No improvement ({epochs_without_improvement}/{early_stopping_patience} epochs).")
        if epochs_without_improvement >= early_stopping_patience:
            print("Early stopping triggered.")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"Restored best model (val accuracy = {best_val_accuracy:.4f}).")

print("")
print("Training complete!")

In [ ]:
# =============================================================================
# SAVE HIERARCHICAL MODEL — Local + Drive/Kaggle + HF Hub
# =============================================================================
# HierarchicalClassifier is a custom nn.Module, so we save:
#   1. encoder/        → HF format (load with AutoModel.from_pretrained)
#   2. tokenizer/      → HF format
#   3. hierarchical_head.pt    → state_dict for attention + classifier layers
#   4. hierarchical_config.json → params needed to reconstruct the wrapper
#
# Loading: instantiate HierarchicalClassifier with base_model_name, then
# load encoder via from_pretrained() and the head via load_state_dict().
# =============================================================================

import os
import json

HF_USERNAME = "sloppydomain"
HF_REPO_NAME = "cjpe-incaselawbert-hierarchical"   # ← new repo for hierarchical model
HF_PRIVATE = True
PUSH_TO_HF = True
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

artifact_subdir = model_name.split('/')[-1] + '_hierarchical_final'

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None or os.path.exists('/kaggle/working')
IS_COLAB = False
if not IS_KAGGLE:
    try:
        import google.colab
        IS_COLAB = True
    except (ImportError, ModuleNotFoundError):
        pass

if IS_KAGGLE:
    output_dir = f'/kaggle/working/cjpe_models/{artifact_subdir}/'
    print(f"Platform: KAGGLE — {output_dir}")
elif IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    output_dir = f'/content/drive/MyDrive/cjpe_models/{artifact_subdir}/'
    print(f"Platform: COLAB — {output_dir}")
else:
    output_dir = f'./artifacts/{artifact_subdir}/'
    print(f"Platform: LOCAL — {output_dir}")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# --- Save encoder via HF API (so AutoModel.from_pretrained works) ---
print(f"\n[1/4] Saving encoder (HF format) to {output_dir}")
model.encoder.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# --- Save hierarchical head (attention + classifier) ---
print(f"[2/4] Saving hierarchical head weights")
head_state = {
    'chunk_attention.weight': model.chunk_attention.weight.detach().cpu(),
    'chunk_attention.bias': model.chunk_attention.bias.detach().cpu(),
    'classifier.weight': model.classifier.weight.detach().cpu(),
    'classifier.bias': model.classifier.bias.detach().cpu(),
}
torch.save(head_state, os.path.join(output_dir, 'hierarchical_head.pt'))

# --- Save config for reconstruction ---
hierarchical_config = {
    'model_type': 'hierarchical',
    'base_model_name': model_name,
    'hidden_size': model.hidden_size,
    'num_labels': model.num_labels,
    'max_chunks': MAX_CHUNKS,
    'chunk_size': CHUNK_SIZE,
    'dropout_prob': 0.1,
    'best_val_accuracy': float(best_val_accuracy),
    'class_weights': class_weights_np.tolist(),
    'class_distribution': class_distribution,
    'labels': {0: 'rejected', 1: 'accepted'},
    'trained_on_platform': 'KAGGLE' if IS_KAGGLE else ('COLAB' if IS_COLAB else 'LOCAL'),
    'training_data': '37,387 docs (multi_train + single_train)',
    'truncation_strategy': f'hierarchical: {MAX_CHUNKS} chunks × {CHUNK_SIZE} tokens',
}
with open(os.path.join(output_dir, 'hierarchical_config.json'), 'w') as f:
    json.dump(hierarchical_config, f, indent=2)

print(f"✓ Local save complete")
print(f"  Files: pytorch_model.bin (encoder), tokenizer files, hierarchical_head.pt, hierarchical_config.json")

# --- Push to HF Hub ---
if PUSH_TO_HF:
    print(f"\n[3/4] Pushing to HF Hub: {HF_REPO_ID}")
    try:
        from huggingface_hub import HfApi, create_repo

        api = HfApi()
        create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True, repo_type="model")
        print(f"  Repo ready: https://huggingface.co/{HF_REPO_ID}")

        api.upload_folder(
            folder_path=output_dir,
            repo_id=HF_REPO_ID,
            repo_type="model",
            commit_message=f"Upload hierarchical {model_name} (val_acc={best_val_accuracy:.4f})",
        )
        print(f"✓ Pushed to https://huggingface.co/{HF_REPO_ID}")

        # Model card with HIERARCHICAL loading instructions
        model_card = f"""---
language: en
license: mit
tags:
- legal
- classification
- court-judgment-prediction
- indian-law
- hierarchical
- {model_name.split('/')[-1].lower()}
datasets:
- Exploration-Lab/IL-TUR
base_model: {model_name}
metrics:
- accuracy
- f1
---

# CJPE — Hierarchical {model_name.split('/')[-1]}

Hierarchical attention model with `{model_name}` encoder, fine-tuned on the
[IL-TUR CJPE](https://huggingface.co/datasets/Exploration-Lab/IL-TUR) dataset.

Unlike standard BERT classifiers that truncate to 512 tokens, this model splits
each document into up to {MAX_CHUNKS} chunks of {CHUNK_SIZE} tokens each, encodes
each chunk independently, then uses learned attention pooling to aggregate
chunk representations into a single document embedding.

## Performance
- **Best validation accuracy:** {best_val_accuracy:.4f}
- **Class distribution (train):** {class_distribution}
- **Training data:** 37,387 docs (multi_train + single_train combined)
- **Max document coverage:** {MAX_CHUNKS * CHUNK_SIZE} tokens (~{MAX_CHUNKS}× more than standard BERT)

## Loading
This is a CUSTOM model — `from_pretrained` won't work directly. Use:

```python
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import hf_hub_download
import json

REPO = "{HF_REPO_ID}"

class HierarchicalClassifier(nn.Module):
    def __init__(self, base_model_name, num_labels=2, dropout_prob=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.encoder.config.hidden_size
        self.hidden_size = hidden_size
        self.num_labels = num_labels
        self.chunk_attention = nn.Linear(hidden_size, 1)
        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden_size, num_labels)
    def forward(self, input_ids, attention_mask, chunk_mask):
        B, K, L = input_ids.shape
        out = self.encoder(input_ids.view(B*K, L), attention_mask=attention_mask.view(B*K, L))
        cls = out.last_hidden_state[:, 0, :].view(B, K, self.hidden_size)
        s = self.chunk_attention(cls).squeeze(-1).masked_fill(chunk_mask == 0, float('-inf'))
        w = torch.softmax(s, dim=-1)
        doc = (cls * w.unsqueeze(-1)).sum(dim=1)
        return self.classifier(self.dropout(doc))

# Load
config_path = hf_hub_download(repo_id=REPO, filename="hierarchical_config.json")
head_path = hf_hub_download(repo_id=REPO, filename="hierarchical_head.pt")
with open(config_path) as f:
    cfg = json.load(f)

model = HierarchicalClassifier(REPO, cfg['num_labels'], cfg['dropout_prob'])
model.encoder = AutoModel.from_pretrained(REPO)   # encoder weights from repo root
model.load_state_dict(torch.load(head_path), strict=False)  # head weights
tokenizer = AutoTokenizer.from_pretrained(REPO)
model.eval()
```

## Architecture
- Base encoder: {model_name} (~110M params)
- Chunk attention: Linear({model.hidden_size} → 1)
- Classifier: Dropout(0.1) → Linear({model.hidden_size} → 2)
- Total: ~110M params (head adds ~1K)

## Training Details
- Epochs: 10 (with early stopping)
- LR: 5e-6 with linear warmup (1000 steps)
- Batch size: 2 with gradient accumulation 12× (effective 24)
- Mixed precision: fp16
- Loss: CrossEntropyLoss with class weights + label_smoothing=0.1
- Chunking: {MAX_CHUNKS} chunks × {CHUNK_SIZE} tokens (no overlap)
"""
        api.upload_file(
            path_or_fileobj=model_card.encode('utf-8'),
            path_in_repo="README.md",
            repo_id=HF_REPO_ID,
            repo_type="model",
            commit_message="Add model card with hierarchical loading code",
        )
        print(f"[4/4] ✓ Model card uploaded")

    except Exception as e:
        print(f"⚠ HF Hub push failed: {e}")
        print(f"  Model is still saved locally at {output_dir}")
else:
    print(f"\n[3/4] HF Hub push skipped (PUSH_TO_HF=False)")
    print(f"[4/4] Done")

In [ ]:
# =============================================================================
# TEST DATA PREP — chunk-based, same shape as train (4-tuple TensorDataset)
# =============================================================================

# Chunk test docs (~30s for 1517 docs)
test_input_ids, test_attention_masks, test_n_chunks = chunk_dataset(test_set, tokenizer, "test")
test_chunk_masks_np = n_chunks_to_chunk_mask(test_n_chunks, MAX_CHUNKS)
test_labels = test_set['label'].to_numpy().astype(int)

# Tensors
prediction_inputs = torch.tensor(test_input_ids, dtype=torch.long)
prediction_masks = torch.tensor(test_attention_masks, dtype=torch.long)
prediction_chunk_masks = torch.tensor(test_chunk_masks_np, dtype=torch.long)
prediction_labels = torch.tensor(test_labels, dtype=torch.long)

# DataLoader — sequential for ordered metrics
prediction_data = TensorDataset(prediction_inputs, prediction_masks,
                                prediction_chunk_masks, prediction_labels)
prediction_sampler = SequentialSampler(prediction_data)
prediction_dataloader = DataLoader(prediction_data, sampler=prediction_sampler, batch_size=batch_size)

print(f"✓ Test set chunked: {len(prediction_inputs)} docs, "
      f"shape {tuple(prediction_inputs.shape)}, "
      f"{len(prediction_dataloader)} batches")

In [ ]:
print('Predicting labels for {:,} test sentences...'.format(len(prediction_inputs)))
model.eval()

predictions, true_labels = [], []

# Hierarchical inference under fp16 autocast
for step, batch in enumerate(prediction_dataloader):
    batch = tuple(t.to(device) for t in batch)
    b_input_ids, b_input_mask, b_chunk_mask, b_labels = batch

    with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
        logits = model(b_input_ids, attention_mask=b_input_mask, chunk_mask=b_chunk_mask)

    logits = logits.detach().cpu().tolist()
    label_ids = b_labels.to('cpu').tolist()

    predictions.append(logits)
    true_labels.append(label_ids)

print('    DONE.')

In [ ]:
predictions = np.concatenate(predictions, axis=0)
true_labels = np.concatenate(true_labels, axis=0)
pred_flat = np.argmax(predictions, axis=1).flatten()
labels_flat = true_labels.flatten()

flat_accuracy(predictions,true_labels)

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import csv
import tensorflow as tf
import torch
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
# from google.colab import drive  # local run
from sklearn.metrics import f1_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
import textwrap
import progressbar
import keras
from keras.preprocessing.sequence import pad_sequences
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertForSequenceClassification, BertConfig
from transformers import get_linear_schedule_with_warmup
import time
import datetime
import random

In [ ]:
def metrics_calculator(preds, test_labels):
    cm = confusion_matrix(test_labels, preds)
    TP = []
    FP = []
    FN = []
    for i in range(0,2):
        summ = 0
        for j in range(0,2):
            if(i!=j):
                summ=summ+cm[i][j]

        FN.append(summ)
    for i in range(0,2):
        summ = 0
        for j in range(0,2):
            if(i!=j):
                summ=summ+cm[j][i]

        FP.append(summ)
    for i in range(0,2):
        TP.append(cm[i][i])
    precision = []
    recall = []
    for i in range(0,2):
        precision.append(TP[i]/(TP[i] + FP[i]))
        recall.append(TP[i]/(TP[i] + FN[i]))

    macro_precision = sum(precision)/2
    macro_recall = sum(recall)/2
    micro_precision = sum(TP)/(sum(TP) + sum(FP))
    micro_recall = sum(TP)/(sum(TP) + sum(FN))
    micro_f1 = (2*micro_precision*micro_recall)/(micro_precision + micro_recall)
    macro_f1 = (2*macro_precision*macro_recall)/(macro_precision + macro_recall)
    return macro_precision, macro_recall, macro_f1, micro_precision, micro_recall, micro_f1

# --- TEST SET METRICS ---
macro_precision, macro_recall, macro_f1, micro_precision, micro_recall, micro_f1 = metrics_calculator(pred_flat, labels_flat)

print("=" * 50)
print("TEST SET METRICS")
print("=" * 50)
print(f"  Macro Precision: {macro_precision:.4f}")
print(f"  Macro Recall:    {macro_recall:.4f}")
print(f"  Macro F1:        {macro_f1:.4f}")
print(f"  Micro Precision: {micro_precision:.4f}")
print(f"  Micro Recall:    {micro_recall:.4f}")
print(f"  Micro F1:        {micro_f1:.4f}")
print("=" * 50)

In [ ]:
# Reassign prediction_dataloader to VALIDATION set for the next inference cell
prediction_data = TensorDataset(validation_inputs, validation_masks,
                                validation_chunk_masks_t, validation_labels)
prediction_sampler = SequentialSampler(prediction_data)
prediction_dataloader = DataLoader(prediction_data, sampler=prediction_sampler, batch_size=batch_size)

prediction_inputs = validation_inputs  # keep print statement working

print(f"✓ prediction_dataloader reassigned to validation: "
      f"{len(prediction_inputs)} docs, {len(prediction_dataloader)} batches")

In [ ]:
print('Predicting labels for {:,} validation sentences...'.format(len(prediction_inputs)))
model.eval()

predictions, true_labels = [], []

for step, batch in enumerate(prediction_dataloader):
    batch = tuple(t.to(device) for t in batch)
    b_input_ids, b_input_mask, b_chunk_mask, b_labels = batch

    with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
        logits = model(b_input_ids, attention_mask=b_input_mask, chunk_mask=b_chunk_mask)

    logits = logits.detach().cpu().tolist()
    label_ids = b_labels.to('cpu').tolist()

    predictions.append(logits)
    true_labels.append(label_ids)

print('    DONE.')

In [ ]:
predictions = np.concatenate(predictions, axis=0)
true_labels = np.concatenate(true_labels, axis=0)
pred_flat = np.argmax(predictions, axis=1).flatten()
labels_flat = true_labels.flatten()

flat_accuracy(predictions,true_labels)

In [ ]:
# --- VALIDATION SET METRICS ---
macro_precision, macro_recall, macro_f1, micro_precision, micro_recall, micro_f1 = metrics_calculator(pred_flat, labels_flat)

print("=" * 50)
print("VALIDATION SET METRICS")
print("=" * 50)
print(f"  Macro Precision: {macro_precision:.4f}")
print(f"  Macro Recall:    {macro_recall:.4f}")
print(f"  Macro F1:        {macro_f1:.4f}")
print(f"  Micro Precision: {micro_precision:.4f}")
print(f"  Micro Recall:    {micro_recall:.4f}")
print(f"  Micro F1:        {micro_f1:.4f}")
print("=" * 50)

In [ ]:
# =============================================================================
# PREDICTION HELPER — single-document inference for hierarchical model
# =============================================================================

def predict_judgment(text, model, tokenizer, device, use_amp=True):
    """
    Predict accepted/rejected for a single legal document using the
    hierarchical model. Returns dict with prediction, confidence, attention
    weights showing which chunks the model focused on.
    """
    model.eval()

    # Chunk the input text using the same logic as training
    chunks_ids, chunks_mask, n_real = make_chunks(text, tokenizer)

    # Build chunk-validity mask (1 for real chunks, 0 for padding chunks)
    chunk_mask = [1] * n_real + [0] * (MAX_CHUNKS - n_real)

    # Convert to tensors — batch dim = 1
    input_ids_t = torch.tensor([chunks_ids], dtype=torch.long).to(device)
    attention_mask_t = torch.tensor([chunks_mask], dtype=torch.long).to(device)
    chunk_mask_t = torch.tensor([chunk_mask], dtype=torch.long).to(device)

    with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp and torch.cuda.is_available()):
        logits = model(input_ids_t, attention_mask=attention_mask_t, chunk_mask=chunk_mask_t)
        probabilities = torch.softmax(logits, dim=-1)

    logits_list = logits[0].float().cpu().tolist()
    probs_list = probabilities[0].float().cpu().tolist()
    pred_idx = int(np.argmax(logits_list))

    return {
        'prediction': 'accepted' if pred_idx == 1 else 'rejected',
        'confidence': round(probs_list[pred_idx], 4),
        'logits': [round(x, 4) for x in logits_list],
        'probabilities': [round(x, 4) for x in probs_list],
        'n_chunks_used': n_real,
    }


# --- Sanity check: warn if model isn't trained yet ---
_is_trained = ('loss_values' in dir() and len(loss_values) > 0) or \
              ('best_val_accuracy' in dir() and best_val_accuracy > 0)

if not _is_trained:
    print("⚠ WARNING: Model does not appear to be trained yet.")
    print("  predict_judgment() will return near-random predictions.")
    print("  Run cell-18 (training loop) first, then re-run this cell.\n")


# --- DEMO: Test on a sample from the test set ---
sample_text = test_set['text'].iloc[0]
sample_label = test_set['label'].iloc[0]
true_label_str = 'accepted' if sample_label == 1 else 'rejected'

result = predict_judgment(sample_text, model, tokenizer, device)

print(f"Sample case ID: {test_set['id'].iloc[0]}")
print(f"Doc length: ~{len(sample_text)} chars, used {result['n_chunks_used']} chunks")
print(f"Text preview: {sample_text[:300]}...")
print(f"\n--- Prediction ---")
print(f"  True label:   {true_label_str}")
print(f"  Predicted:    {result['prediction']}")
print(f"  Confidence:   {result['confidence']:.2%}")
print(f"  P(rejected):  {result['probabilities'][0]:.4f}")
print(f"  P(accepted):  {result['probabilities'][1]:.4f}")
print(f"  Match:        {'✓' if result['prediction'] == true_label_str else '✗'}")